# Capítulo 12 — Segurança: protegendo suas riquezas

**Livro:** começa na página **438** do PDF.

**Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")
from banco import preparar
motor = preparar(12, "gabarito", recomecar=True)  # preparar(..., recomecar=True) volta o capítulo ao início
%reload_ext sql
%config SqlMagic.displaylimit = 60
%config SqlMagic.feedback = 0
%sql motor

## Proteja a conta de usuários raiz · PDF p. 441

#### Exercício: Quem é você no banco

Mostre o usuário atual e se ele é superusuário (`pg_roles.rolsuper`). Liste também os papéis que não são do sistema (nome sem `pg_`). O usuário `estudos` é o "root" deste banco.

In [ ]:
%%sql
SELECT rolname, rolsuper, rolcanlogin
FROM pg_roles WHERE rolname NOT LIKE 'pg%' ORDER BY rolname;

## Adicione um novo usuário · PDF p. 442

#### Exercício: CREATE ROLE ... LOGIN

Crie o usuário `estagiaria` (`CREATE ROLE estagiaria LOGIN`; num servidor de verdade você daria também uma senha). Confira que ele ainda não pode ler `palhacos_info` (`has_table_privilege`).

In [ ]:
%%sql
CREATE ROLE estagiaria LOGIN;
SELECT has_table_privilege('estagiaria', 'palhacos_info', 'SELECT') AS pode_ler;

## Um simples comando GRANT · PDF p. 443

#### Exercício: GRANT SELECT

Dê à `estagiaria` permissão de leitura em `palhacos_info`. Depois "vire" a estagiária com `SET ROLE`, conte as linhas e volte com `RESET ROLE`.

In [ ]:
%%sql
GRANT SELECT ON palhacos_info TO estagiaria;
SET ROLE estagiaria;
SELECT current_user, COUNT(*) AS palhacos FROM palhacos_info;

#### Exercício: Testando o que ela não pode

Ainda como `estagiaria`, tente apagar um palhaço (`try/except`). Volte para o seu usuário no `finally`.

In [ ]:
try:
    %sql DELETE FROM palhacos_info WHERE palhaco_id = 1
except Exception as erro:
    print("Sem permissão:", "permission denied" in str(erro) or "permissão negada" in str(erro))
finally:
    %sql RESET ROLE
%sql SELECT current_user

## Variações do comando GRANT · PDF p. 446

#### Exercício: Várias permissões e permissão por coluna

Dê à `estagiaria` `SELECT` e `INSERT` em `info_local`, e `UPDATE` **só** na coluna `descricao` de `palhacos_info`. Confira com `has_table_privilege` e `has_column_privilege`.

In [ ]:
%%sql
GRANT SELECT, INSERT ON info_local TO estagiaria;
GRANT UPDATE (descricao) ON palhacos_info TO estagiaria;
SELECT has_table_privilege('estagiaria', 'info_local', 'INSERT') AS insere_local,
       has_column_privilege('estagiaria', 'palhacos_info', 'descricao', 'UPDATE') AS muda_descricao,
       has_column_privilege('estagiaria', 'palhacos_info', 'nome', 'UPDATE') AS muda_nome;

## REVOKE (REVOGUE) privilégios · PDF p. 447

#### Exercício: REVOKE

Retire o `INSERT` em `info_local` da `estagiaria` e confira.

In [ ]:
%%sql
REVOKE INSERT ON info_local FROM estagiaria;
SELECT has_table_privilege('estagiaria', 'info_local', 'SELECT') AS le,
       has_table_privilege('estagiaria', 'info_local', 'INSERT') AS insere;

## REVOGANDO uma GRANT OPTION usada · PDF p. 447

#### Exercício: WITH GRANT OPTION e CASCADE

Crie `gerente_palhacos` e `visitante`. Dê ao gerente `SELECT` em `atividades` `WITH GRANT OPTION`. Como gerente (`SET ROLE`), repasse ao visitante. Volte (`RESET ROLE`), revogue do gerente com `CASCADE` e veja que o visitante também perdeu.

In [ ]:
%%sql
CREATE ROLE gerente_palhacos;
CREATE ROLE visitante;
GRANT SELECT ON atividades TO gerente_palhacos WITH GRANT OPTION;
SET ROLE gerente_palhacos;
GRANT SELECT ON atividades TO visitante;
RESET ROLE;
REVOKE SELECT ON atividades FROM gerente_palhacos CASCADE;
SELECT has_table_privilege('gerente_palhacos', 'atividades', 'SELECT') AS gerente,
       has_table_privilege('visitante', 'atividades', 'SELECT') AS visitante;

## REVOGANDO com mais precisão · PDF p. 448

#### Exercício: RESTRICT

Repita o repasse e agora tente revogar com `RESTRICT` (padrão): como há uma permissão dependente, o banco recusa (`try/except`). Depois revogue só a opção de repassar: `REVOKE GRANT OPTION FOR ... CASCADE`.

In [ ]:
%sql GRANT SELECT ON atividades TO gerente_palhacos WITH GRANT OPTION
%sql SET ROLE gerente_palhacos
%sql GRANT SELECT ON atividades TO visitante
%sql RESET ROLE
try:
    %sql REVOKE SELECT ON atividades FROM gerente_palhacos RESTRICT
except Exception as erro:
    print("RESTRICT recusou: existem permissões dependentes")
%sql REVOKE GRANT OPTION FOR SELECT ON atividades FROM gerente_palhacos CASCADE
%sql SELECT has_table_privilege('gerente_palhacos', 'atividades', 'SELECT') AS gerente_ainda_le, has_table_privilege('gerente_palhacos', 'atividades', 'SELECT WITH GRANT OPTION') AS pode_repassar

## Utilizando sua role · PDF p. 452

#### Exercício: Papéis (roles) de grupo

Crie o papel `equipe_dados` com `SELECT` em `palhacos_info` e `info_local`. Crie o usuário `leitor_relatorios` (LOGIN) e coloque-o no papel (`GRANT equipe_dados TO leitor_relatorios`). Ele herdou a leitura?

In [ ]:
%%sql
CREATE ROLE equipe_dados;
GRANT SELECT ON palhacos_info, info_local TO equipe_dados;
CREATE ROLE leitor_relatorios LOGIN;
GRANT equipe_dados TO leitor_relatorios;
SELECT has_table_privilege('leitor_relatorios', 'info_local', 'SELECT') AS le_local,
       has_table_privilege('leitor_relatorios', 'atividades', 'SELECT') AS le_atividades;

## Eliminando roles · PDF p. 453

#### Exercício: DROP ROLE

Elimine o papel `visitante`. Antes, tire as permissões dele com `DROP OWNED BY visitante` (o banco não deixa apagar um papel que ainda tem permissões).

In [ ]:
%%sql
DROP OWNED BY visitante;
DROP ROLE visitante;
SELECT rolname FROM pg_roles WHERE rolname = 'visitante';

## Utilizando sua role WITH ADMIN OPTION · PDF p. 454

#### Exercício: WITH ADMIN OPTION

Crie `chefe_equipe` e dê a ele o papel `equipe_dados` `WITH ADMIN OPTION`: ele pode colocar outras pessoas no papel. Como chefe, coloque a `estagiaria`. Confira em `pg_auth_members`.

In [ ]:
%%sql
CREATE ROLE chefe_equipe;
GRANT equipe_dados TO chefe_equipe WITH ADMIN OPTION;
SET ROLE chefe_equipe;
GRANT equipe_dados TO estagiaria;
RESET ROLE;
SELECT r.rolname AS papel, m.rolname AS membro, a.admin_option
FROM pg_auth_members a
JOIN pg_roles r ON r.oid = a.roleid
JOIN pg_roles m ON m.oid = a.member
WHERE r.rolname = 'equipe_dados'
ORDER BY membro;

## Combinando CREATE USER com GRANT · PDF p. 458

#### Exercício: Criar o usuário já no papel

No MySQL dá para criar e dar permissões de uma vez. No PostgreSQL: `CREATE USER novato IN ROLE equipe_dados` (`CREATE USER` = `CREATE ROLE ... LOGIN`). Confira que ele lê `palhacos_info`.

In [ ]:
%%sql
CREATE USER novato IN ROLE equipe_dados;
SELECT has_table_privilege('novato', 'palhacos_info', 'SELECT') AS novato_le;

## A lista de Greg agora é global! · PDF p. 459

#### Exercício: Relatório de permissões

Monte o relatório de quem pode o quê nas tabelas deste esquema (`information_schema.role_table_grants`), sem o seu próprio usuário.

In [ ]:
%%sql
SELECT grantee AS quem, table_name AS tabela, string_agg(privilege_type, ', ' ORDER BY privilege_type) AS pode
FROM information_schema.role_table_grants
WHERE table_schema = current_schema() AND grantee <> current_user
GROUP BY grantee, table_name
ORDER BY grantee, table_name;